# Apartment Price Prediction – Iterative Model Training

This notebook trains a regression model to predict monthly rental prices for apartments in the Canton of Zurich.
We perform **two iterations** of model training and document the results.

**New feature introduced in Iteration 2:** `area_per_room` – average room size in m² (not used in prior exercises)

After running all cells, copy the printed R² values into `readme.md`.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

print('Libraries loaded successfully')

## Load and Explore Data

In [ ]:
# Load the original apartment dataset
df_orig = pd.read_csv('../../week1/original_apartment_data_analytics_hs24.csv', sep=',', encoding='utf-8')

# Basic preprocessing: remove missing values and duplicates
df = df_orig.dropna().drop_duplicates().reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
print('Target variable (price) distribution:')
print(df['price'].describe().round(0))
df[['rooms', 'area', 'price', 'pop', 'tax_income']].head()

---
## Iteration 1: Baseline Model

**Objective:** Establish a performance baseline using the 7 original features from prior exercises.

**Preprocessing steps:**
- Drop rows with missing values
- Remove duplicate rows
- StandardScaler for feature normalization (inside pipeline)
- 5-fold cross-validation

**Models tried:**
1. Linear Regression (baseline)
2. Random Forest (n_estimators=100, random_state=42)

In [ ]:
FEATURES_V1 = ['rooms', 'area', 'pop', 'pop_dens', 'frg_pct', 'emp', 'tax_income']
TARGET = 'price'

X1 = df[FEATURES_V1]
y1 = df[TARGET]

# Model 1: Linear Regression
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
lr_cv = cross_val_score(lr_pipe, X1, y1, cv=5, scoring='r2')

# Model 2: Random Forest
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])
rf_cv = cross_val_score(rf_pipe, X1, y1, cv=5, scoring='r2')

print('=== ITERATION 1 RESULTS (5-Fold CV) ===')
print(f'Linear Regression:      R² = {lr_cv.mean():.4f}  std = {lr_cv.std():.4f}')
print(f'Random Forest (n=100):  R² = {rf_cv.mean():.4f}  std = {rf_cv.std():.4f}')
print()
print('Observation: Random Forest likely overfits (high train R², lower CV R²).')
print('Best model in Iter 1:', 'Random Forest' if rf_cv.mean() > lr_cv.mean() else 'Linear Regression')

---
## Iteration 2: New Feature + Improved Models

**Objective:** Improve generalization by adding a new engineered feature and using better-tuned models.

**New feature: `area_per_room`**
- Computed as: `area / rooms`
- Represents average room size in m²
- **Not used in prior exercises** (Week 1 and Week 2 used area and rooms separately)
- Captures apartment quality: a 3-room apartment with 90 m² has much larger rooms than one with 60 m²

**Preprocessing steps:**
- All steps from Iteration 1
- New engineered feature: `area_per_room = area / rooms`
- StandardScaler applied to all 8 features
- 5-fold cross-validation

**Models tried:**
1. Ridge Regression (alpha=10) – regularized linear model to reduce overfitting
2. Gradient Boosting (n_estimators=200, max_depth=4, learning_rate=0.05)

In [ ]:
# Engineer the new feature
df['area_per_room'] = df['area'] / df['rooms']

FEATURES_V2 = FEATURES_V1 + ['area_per_room']

X2 = df[FEATURES_V2]
y2 = df[TARGET]

# Model 1: Ridge Regression (regularized, avoids overfitting)
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10.0))
])
ridge_cv = cross_val_score(ridge_pipe, X2, y2, cv=5, scoring='r2')

# Model 2: Gradient Boosting (sequential ensemble of shallow trees)
gb_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    ))
])
gb_cv = cross_val_score(gb_pipe, X2, y2, cv=5, scoring='r2')

print('=== ITERATION 2 RESULTS (5-Fold CV) ===')
print(f'Ridge (alpha=10):           R² = {ridge_cv.mean():.4f}  std = {ridge_cv.std():.4f}')
print(f'Gradient Boosting (n=200):  R² = {gb_cv.mean():.4f}  std = {gb_cv.std():.4f}')
print()
print('Best model in Iter 2:', 'Gradient Boosting' if gb_cv.mean() > ridge_cv.mean() else 'Ridge')

## Results Summary

Copy these values into `readme.md` to complete the documentation.

In [ ]:
print('=' * 70)
print(f'{"Iter":<6} {"Model":<35} {"CV R² Mean":<12} {"CV R² Std"}')
print('=' * 70)
print(f'{"1":<6} {"Linear Regression":<35} {lr_cv.mean():<12.4f} {lr_cv.std():.4f}')
print(f'{"1":<6} {"Random Forest (n=100)":<35} {rf_cv.mean():<12.4f} {rf_cv.std():.4f}')
print(f'{"2":<6} {"Ridge (alpha=10)":<35} {ridge_cv.mean():<12.4f} {ridge_cv.std():.4f}')
print(f'{"2":<6} {"Gradient Boosting (n=200)":<35} {gb_cv.mean():<12.4f} {gb_cv.std():.4f}')
print('=' * 70)
print()
best_v1 = max(lr_cv.mean(), rf_cv.mean())
best_v2 = max(ridge_cv.mean(), gb_cv.mean())
print(f'Best Iteration 1 R²: {best_v1:.4f}')
print(f'Best Iteration 2 R²: {best_v2:.4f}')
print(f'Improvement:         +{best_v2 - best_v1:.4f}')

## Train Final Model and Save

We train **Gradient Boosting** (best performer from Iteration 2) on the full dataset and save it as `apartment_model.pkl`.

In [ ]:
# Train final model on full dataset
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    ))
])
final_model.fit(X2, y2)

# Save model to disk
with open('apartment_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print('Model saved: apartment_model.pkl')
print(f'Features used ({len(FEATURES_V2)}): {FEATURES_V2}')
print()
print('Done! You can now run the Gradio app with: python app.py')